# Controlled split comparison and repeated patient-level CV


| RUN | What it does | Hybrid trainings |
|---|---|---|
| `patient_r0` | Repeat 1 of patient-level 5-fold CV over all 320 patients | 20 |
| `patient_r1` | Repeat 2 (different fold assignment) | 20 |
| `patient_r2` | Repeat 3 | 20 |
| `slice` | Same pipelines with a random slice-level 5-fold split | 15 |


1. Every model, under both splits, uses the same recipe, the same 80% fit set and the same metrics, so Table I becomes a controlled comparison.
2. No early stopping: each network trains on the fixed two-phase schedule (8 + 15 epochs). Validation patients are therefore never used to train or stop a model.
3. Inside each outer training fold, 20% of patients form a calibration set. It is used only to tune T1 and T2, choose the stage-2 model and choose the single-stage backbone. Test-fold patients are not used for any choice.
4. T1 is tuned for the macro-F1 of the whole cascade (after T2 and the stage-2 model are fixed on CN and MCI calibration patients).
5. One network per task and fold, so there is no fold ensemble and no weighting by slice accuracy.

In [ ]:
import os, json, math, random, shutil, time, pickle, warnings
import numpy as np
import pandas as pd
from collections import defaultdict
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (resnet18, ResNet18_Weights, densenet121, DenseNet121_Weights,
                                resnet50, ResNet50_Weights, efficientnet_b0, EfficientNet_B0_Weights)

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, f1_score, roc_auc_score
from scipy.stats import binomtest

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


## Configuration

In [ ]:
RUN = 'patient_r1'

INPUT_ROOT = '/kaggle/input'
WORK_DIR   = '/kaggle/working'

DATA_ROOT = None
for root, dirs, _ in os.walk(INPUT_ROOT):
    if os.path.basename(root) == 'adni_nifti_preprocessed' and {'AD', 'CN', 'MCI'} <= set(dirs):
        DATA_ROOT = root
        break
assert DATA_ROOT, 'Attach the ADNI dataset (a folder adni_nifti_preprocessed with AD, CN and MCI inside)'
OUT_DIR = os.path.join(WORK_DIR, 'results_' + RUN)
os.makedirs(OUT_DIR, exist_ok=True)

MODE   = 'slice' if RUN == 'slice' else 'patient'
REPEAT = int(RUN.split('_r')[1]) if MODE == 'patient' else 0
SEED   = 42
CV_SEED = 100 + REPEAT if MODE == 'patient' else 200

N_OUTER      = 5
CAL_FRACTION = 0.20
IMAGE_SIZE   = 224
BATCH_SIZE   = 32
NUM_WORKERS  = 2
VALID_EXTS   = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

# Hybrid recipe
PRETRAINED      = True
PHASE_A_EPOCHS  = 8
PHASE_B_EPOCHS  = 15
WARMUP_EPOCHS   = 2
LR_MIN          = 1e-6
LR_HEAD_A       = 3e-3
LR_HEAD_B       = 1e-3
LR_BACKBONE_B   = 1e-4
WEIGHT_DECAY    = 5e-5
GRAD_CLIP_NORM  = 1.0
HEAD_HIDDEN     = 384
DROPOUT         = 0.30
LABEL_SMOOTHING = 0.05
RANDAUG_N, RANDAUG_M = 2, 7
RANDOM_ERASING  = 0.20

BACKBONES      = ['resnet18+densenet121', 'resnet50', 'densenet121', 'efficientnet_b0']
S2_CLASSIFIERS = ['logreg', 'linear_svm', 'rbf_svm', 'random_forest', 'mlp']
PCA_DIM        = 256
N_BOOT         = 2000
THRESHOLDS     = np.round(np.arange(0.05, 0.951, 0.01), 2)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
CLASS_NAMES = ['AD', 'CN', 'MCI']
AD, CN, MCI = 0, 1, 2
TASKS = {'AD_vs_REST': ([AD], [AD, CN, MCI]), 'CN_vs_REST': ([CN], [AD, CN, MCI]),
         'MCI_vs_REST': ([MCI], [AD, CN, MCI]), 'CN_vs_MCI': ([MCI], [CN, MCI])}
OVA_TASKS = ['AD_vs_REST', 'CN_vs_REST', 'MCI_vs_REST']

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(SEED)
print('RUN:', RUN, '| data:', DATA_ROOT)

RUN: patient_r1 | data: /kaggle/input/datasets/malakkhaled202300369/adni-data/adni_nifti_preprocessed


## Patients and slices (patients listed under more than one class are excluded, as before)

In [ ]:
patient_classes = defaultdict(set)
patient_files = defaultdict(list)
for cls in CLASS_NAMES:
    cls_dir = os.path.join(DATA_ROOT, cls)
    for pid in sorted(os.listdir(cls_dir)):
        pdir = os.path.join(cls_dir, pid)
        if not os.path.isdir(pdir):
            continue
        files = sorted(os.path.join(pdir, f) for f in os.listdir(pdir) if os.path.splitext(f)[1].lower() in VALID_EXTS)
        if files:
            patient_classes[pid].add(cls)
            patient_files[(pid, cls)] = files

patients = sorted(p for p, c in patient_classes.items() if len(c) == 1)
patient_label = {p: CLASS_NAMES.index(next(iter(patient_classes[p]))) for p in patients}

slice_path, slice_pid, slice_label = [], [], []
for p in patients:
    for f in patient_files[(p, CLASS_NAMES[patient_label[p]])]:
        slice_path.append(f)
        slice_pid.append(p)
        slice_label.append(patient_label[p])
slice_pid = np.array(slice_pid)
slice_label = np.array(slice_label)
n_slices = len(slice_path)

pid_idx = defaultdict(list)
for i, p in enumerate(slice_pid):
    pid_idx[p].append(i)
pid_idx = {p: np.array(v) for p, v in pid_idx.items()}

def idx_of(pids):
    return np.concatenate([pid_idx[p] for p in pids])

counts = {c: sum(1 for p in patients if patient_label[p] == i) for i, c in enumerate(CLASS_NAMES)}
print(f'Patients: {len(patients)} {counts} | excluded: {len(patient_classes) - len(patients)} | slices: {n_slices}')

Patients: 320 {'AD': 54, 'CN': 88, 'MCI': 178} | excluded: 0 | slices: 8512


## Load all slices into memory once (uint8, about 0.4 GB)

In [ ]:
IMAGES = np.zeros((n_slices, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
for i, f in enumerate(slice_path):
    IMAGES[i] = np.array(Image.open(f).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS))

normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
hflip = transforms.RandomHorizontalFlip(p=0.5)
rand_aug = transforms.RandAugment(num_ops=RANDAUG_N, magnitude=RANDAUG_M)
jitter = transforms.ColorJitter(brightness=0.15, contrast=0.15)
erasing = transforms.RandomErasing(p=RANDOM_ERASING, scale=(0.02, 0.12))

class SliceSet(Dataset):
    def __init__(self, idx, labels, augment):
        self.idx = np.asarray(idx)
        self.labels = np.asarray(labels)
        self.augment = augment

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        t = torch.from_numpy(IMAGES[self.idx[i]].copy()).unsqueeze(0).repeat(3, 1, 1)
        if self.augment:
            t = rand_aug(hflip(t))
            t = erasing(normalize(jitter(t.float() / 255.0)))
        else:
            t = normalize(t.float() / 255.0)
        return t, int(self.labels[i])

def make_loader(idx, labels, augment, shuffle):
    return DataLoader(SliceSet(idx, labels, augment), batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))
print('Slices in memory:', IMAGES.shape)

Slices in memory: (8512, 224, 224)


## Hybrid network and fixed two-phase schedule

In [ ]:
def make_head(in_dim, num_classes):
    return nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(in_dim, HEAD_HIDDEN), nn.BatchNorm1d(HEAD_HIDDEN), nn.GELU(),
                         nn.Dropout(DROPOUT * 0.5), nn.Linear(HEAD_HIDDEN, num_classes))

class ResNetDenseNetHybrid(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        r = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        r.fc = nn.Identity()
        d = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        d.classifier = nn.Identity()
        self.resnet, self.densenet = r, d
        self.head = make_head(512 + 1024, num_classes)

    def forward(self, x):
        return self.head(torch.cat([self.resnet(x), self.densenet(x)], dim=1))

def set_phase(model, phase_b):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True
    if phase_b:
        for block in (model.resnet.layer4, model.densenet.features.denseblock4, model.densenet.features.norm5):
            for p in block.parameters():
                p.requires_grad = True

def set_train_mode(model, phase_b):
    model.train()
    model.resnet.eval()
    model.densenet.eval()
    if phase_b:
        model.resnet.layer4.train(True)
        model.densenet.features.denseblock4.train(True)
        model.densenet.features.norm5.train(True)
    model.head.train(True)

def build_optimizer(model, lr_head, lr_backbone=None):
    head_ids = {id(p) for p in model.head.parameters()}
    groups = [{'params': list(model.head.parameters()), 'lr': lr_head}]
    back = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    if back and lr_backbone:
        groups.append({'params': back, 'lr': lr_backbone})
    return optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

def cosine_with_warmup(optimizer, total_epochs, warmup):
    base = optimizer.param_groups[0]['lr']
    def lr_lambda(epoch):
        if epoch < warmup:
            return (epoch + 1) / max(1, warmup)
        progress = (epoch - warmup) / max(1, total_epochs - warmup)
        return max(LR_MIN / base, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def run_phase(model, loader, criterion, n_epochs, lr_head, lr_backbone, phase_b):
    set_phase(model, phase_b)
    opt = build_optimizer(model, lr_head, lr_backbone)
    sched = cosine_with_warmup(opt, n_epochs, WARMUP_EPOCHS)
    scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))
    for _ in range(n_epochs):
        set_train_mode(model, phase_b)
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                loss = criterion(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP_NORM)
            scaler.step(opt)
            scaler.update()
        sched.step()

def train_hybrid(idx, y, seed):
    set_seed(seed)
    pos = int(y.sum())
    neg = len(y) - pos
    cw = torch.tensor([len(y) / (2.0 * max(neg, 1)), len(y) / (2.0 * max(pos, 1))], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=LABEL_SMOOTHING)
    model = ResNetDenseNetHybrid(2).to(device)
    loader = make_loader(idx, y, augment=True, shuffle=True)
    run_phase(model, loader, criterion, PHASE_A_EPOCHS, LR_HEAD_A, None, phase_b=False)
    run_phase(model, loader, criterion, PHASE_B_EPOCHS, LR_HEAD_B, LR_BACKBONE_B, phase_b=True)
    model.eval()
    return model

def predict_slices(model, idx):
    # positive-class probability per slice, averaging the logits of the slice and its horizontal flip
    loader = make_loader(idx, np.zeros(len(idx), dtype=int), augment=False, shuffle=False)
    out = []
    model.eval()
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(x) + model(torch.flip(x, dims=[-1]))
            out.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)

## Frozen ImageNet features (extracted once per run, no augmentation)

In [ ]:
def build_extractor(name):
    if name == 'resnet18+densenet121':
        r = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        r.fc = nn.Identity()
        d = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        d.classifier = nn.Identity()
        class Both(nn.Module):
            def __init__(self):
                super().__init__()
                self.r, self.d = r, d
            def forward(self, x):
                return torch.cat([self.r(x), self.d(x)], dim=1)
        return Both()
    if name == 'resnet50':
        m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if PRETRAINED else None)
        m.fc = nn.Identity()
        return m
    if name == 'densenet121':
        m = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        m.classifier = nn.Identity()
        return m
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if PRETRAINED else None)
    m.classifier = nn.Identity()
    return m

feats = {}
loader = make_loader(np.arange(n_slices), np.zeros(n_slices, dtype=int), augment=False, shuffle=False)
for name in BACKBONES:
    ext = build_extractor(name).to(device).eval()
    out = []
    with torch.no_grad():
        for x, _ in loader:
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                out.append(ext(x.to(device)).float().cpu().numpy())
    feats[name] = np.concatenate(out)
    del ext
    if device.type == 'cuda':
        torch.cuda.empty_cache()
print('Features:', {k: v.shape[1] for k, v in feats.items()})

def make_clf(name, n_fit):
    pca = PCA(n_components=min(PCA_DIM, n_fit - 1), random_state=SEED)
    if name == 'logreg':
        clf = LogisticRegression(C=0.1, class_weight='balanced', max_iter=3000)
    elif name == 'linear_svm':
        clf = CalibratedClassifierCV(LinearSVC(C=0.01, class_weight='balanced', max_iter=10000), cv=3)
    elif name == 'rbf_svm':
        clf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, class_weight='balanced', random_state=SEED)
    elif name == 'random_forest':
        clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight='balanced_subsample',
                                     n_jobs=-1, random_state=SEED)
    else:
        clf = MLPClassifier(hidden_layer_sizes=(256,), alpha=1e-3, early_stopping=True, max_iter=300, random_state=SEED)
    return make_pipeline(StandardScaler(), pca, clf)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 208MB/s]


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 174MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 248MB/s]


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 189MB/s]


Features: {'resnet18+densenet121': 1536, 'resnet50': 2048, 'densenet121': 1024, 'efficientnet_b0': 1280}


## Folds

Patient mode: stratified 5-fold split of the 320 patients; inside each training fold, a stratified 20% of patients is the calibration set.
Slice mode: stratified 5-fold split of the slices themselves (patients ignored, as in the reproduced studies); 20% of the training slices are held out
so that every model trains on the same fraction of data as in patient mode.

In [ ]:
folds = []
if MODE == 'patient':
    y_pat = np.array([patient_label[p] for p in patients])
    skf = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=CV_SEED)
    for k, (tr, te) in enumerate(skf.split(np.zeros(len(patients)), y_pat)):
        tr_p = [patients[i] for i in tr]
        te_p = [patients[i] for i in te]
        fit_p, cal_p = train_test_split(tr_p, test_size=CAL_FRACTION, stratify=[patient_label[p] for p in tr_p],
                                        random_state=1000 + 10 * REPEAT + k)
        assert not (set(fit_p) & set(cal_p)) and not (set(tr_p) & set(te_p))
        folds.append({'fit': idx_of(fit_p), 'cal': idx_of(cal_p), 'test': idx_of(te_p)})
else:
    skf = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=CV_SEED)
    for k, (tr, te) in enumerate(skf.split(np.zeros(n_slices), slice_label)):
        fit_i, cal_i = train_test_split(tr, test_size=CAL_FRACTION, stratify=slice_label[tr], random_state=2000 + k)
        folds.append({'fit': np.sort(fit_i), 'cal': np.sort(cal_i), 'test': np.sort(te)})

for k, f in enumerate(folds):
    shared = np.isin(slice_pid[f['test']], np.unique(slice_pid[f['fit']])).mean()
    print(f"fold {k+1}: fit {len(f['fit'])} | cal {len(f['cal'])} | test {len(f['test'])} slices | "
          f"test slices whose patient is also in the fit set: {shared*100:.1f}%")

fold 1: fit 5426 | cal 1374 | test 1712 slices | test slices whose patient is also in the fit set: 0.0%
fold 2: fit 5402 | cal 1392 | test 1718 slices | test slices whose patient is also in the fit set: 0.0%
fold 3: fit 5431 | cal 1384 | test 1697 slices | test slices whose patient is also in the fit set: 0.0%
fold 4: fit 5416 | cal 1396 | test 1700 slices | test slices whose patient is also in the fit set: 0.0%
fold 5: fit 5427 | cal 1400 | test 1685 slices | test slices whose patient is also in the fit set: 0.0%


## Train and predict, fold by fold (finished folds are saved and reused)

In [ ]:
previous = {}
for root, _, files in os.walk(INPUT_ROOT):
    if os.path.basename(root) == 'results_' + RUN:
        for name in files:
            if name.startswith('fold') and name.endswith('.pkl'):
                previous[name] = os.path.join(root, name)

fold_results = []
for k, f in enumerate(folds):
    path = os.path.join(OUT_DIR, f'fold{k+1}.pkl')
    if not os.path.exists(path) and f'fold{k+1}.pkl' in previous:
        shutil.copy(previous[f'fold{k+1}.pkl'], path)
    if os.path.exists(path):
        with open(path, 'rb') as fh:
            res = pickle.load(fh)
        assert np.array_equal(res['ev'], np.concatenate([f['cal'], f['test']])), 'saved fold does not match this split'
        fold_results.append(res)
        print(f'fold {k+1}: reused')
        continue

    t0 = time.time()
    ev = np.concatenate([f['cal'], f['test']])
    res = {'ev': ev, 'hybrid': {}, 'lr': {}, 's2': {}}
    tasks = list(TASKS) if MODE == 'patient' else OVA_TASKS
    for t, task in enumerate(tasks):
        pos_cls, include = TASKS[task]
        tr_i = f['fit'][np.isin(slice_label[f['fit']], include)]
        y = np.isin(slice_label[tr_i], pos_cls).astype(int)
        model = train_hybrid(tr_i, y, seed=SEED + 100 * REPEAT + 10 * k + t)
        res['hybrid'][task] = predict_slices(model, ev)
        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    for bb in BACKBONES:
        clf = make_clf('logreg', len(f['fit']))
        clf.fit(feats[bb][f['fit']], slice_label[f['fit']])
        res['lr'][bb] = clf.predict_proba(feats[bb][ev])

    if MODE == 'patient':
        s2_fit = f['fit'][np.isin(slice_label[f['fit']], [CN, MCI])]
        for bb in BACKBONES:
            for cn in S2_CLASSIFIERS:
                clf = make_clf(cn, len(s2_fit))
                clf.fit(feats[bb][s2_fit], (slice_label[s2_fit] == MCI).astype(int))
                res['s2'][bb + '|' + cn] = clf.predict_proba(feats[bb][ev])[:, 1]

    with open(path, 'wb') as fh:
        pickle.dump(res, fh)
    fold_results.append(res)
    print(f'fold {k+1}: done in {(time.time() - t0) / 60:.1f} min')

fold 1: done in 27.0 min
fold 2: done in 26.8 min
fold 3: done in 26.9 min
fold 4: done in 26.9 min
fold 5: done in 26.8 min


## Decisions, metrics and paired tests

In [ ]:
def metrics3(y, pred):
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1, 2], zero_division=0)
    return {'acc': float(accuracy_score(y, pred)), 'macro_P': float(p.mean()), 'macro_R': float(r.mean()),
            'macro_F1': float(f.mean()), 'R_AD': float(r[AD]), 'R_CN': float(r[CN]), 'R_MCI': float(r[MCI])}

def boot_ci(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    rng = np.random.default_rng(SEED)
    f1s, recs = [], []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        _, r, f, _ = precision_recall_fscore_support(y[i], pred[i], labels=[0, 1, 2], zero_division=0)
        f1s.append(f.mean())
        recs.append(r.mean())
    return [float(v) for v in np.percentile(f1s, [2.5, 97.5])], [float(v) for v in np.percentile(recs, [2.5, 97.5])]

def paired(y, a, b):
    y, a, b = np.asarray(y), np.asarray(a), np.asarray(b)
    mf = lambda yy, pp: precision_recall_fscore_support(yy, pp, labels=[0, 1, 2], zero_division=0)[2].mean()
    rng = np.random.default_rng(SEED)
    d = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        d.append(mf(y[i], a[i]) - mf(y[i], b[i]))
    only_a = int(np.sum((a == y) & (b != y)))
    only_b = int(np.sum((b == y) & (a != y)))
    p = binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b > 0 else 1.0
    return {'diff': float(mf(y, a) - mf(y, b)), 'ci': [float(v) for v in np.percentile(d, [2.5, 97.5])],
            'only_a': only_a, 'only_b': only_b, 'mcnemar_p': float(p)}

def best_threshold(score_fn):
    best_t, best_s = 0.5, -1.0
    for t in THRESHOLDS:
        s = score_fn(t)
        if s > best_s + 1e-9 or (abs(s - best_s) <= 1e-9 and abs(t - 0.5) < abs(best_t - 0.5)):
            best_t, best_s = float(t), s
    return best_t, best_s

def cascade(p_ad, p_mci, t1, t2):
    return np.where(p_ad >= t1, AD, np.where(p_mci >= t2, MCI, CN))

def pmean(values, idx):
    # patient-level mean of slice values; returns a DataFrame indexed by patient
    v = np.asarray(values)
    df = pd.DataFrame(v if v.ndim == 2 else v[:, None])
    df['pid'] = slice_pid[idx]
    return df.groupby('pid').mean()

def bin_macro_f1(y, p, t):
    return f1_score(y, (p >= t).astype(int), average='macro', zero_division=0)

In [ ]:
slice_true, slice_pred = [], defaultdict(list)
rows, selection = [], []
slice_split_probs, slice_split_idx = [], []

for k, (f, res) in enumerate(zip(folds, fold_results)):
    ev = res['ev']
    n_cal = len(f['cal'])
    te_slices = ev[n_cal:]

    # slice-level predictions on the test slices (argmax of the three One-vs-All networks, argmax of each logistic regression)
    ova_slice = np.stack([res['hybrid'][t][n_cal:] for t in OVA_TASKS], axis=1)
    slice_true.append(slice_label[te_slices])
    slice_pred['ova'].append(ova_slice.argmax(1))
    for bb in BACKBONES:
        slice_pred['lr_' + bb].append(res['lr'][bb][n_cal:].argmax(1))

    if MODE == 'slice':
        slice_split_probs.append(ova_slice)
        slice_split_idx.append(te_slices)
        continue

    cal_df = {t: pmean(res['hybrid'][t][:n_cal], f['cal'])[0] for t in TASKS}
    te_df = {t: pmean(res['hybrid'][t][n_cal:], te_slices)[0] for t in TASKS}
    cal_p = list(cal_df['AD_vs_REST'].index)
    te_p = list(te_df['AD_vs_REST'].index)
    y_cal = np.array([patient_label[p] for p in cal_p])
    y_te = np.array([patient_label[p] for p in te_p])

    # One-vs-All: argmax of the three patient-level probabilities, no thresholds
    ova_te = np.stack([te_df[t].loc[te_p].values for t in OVA_TASKS], axis=1).argmax(1)

    # single-stage logistic regression on frozen features; backbone chosen on the calibration patients
    lr_cal = {bb: pmean(res['lr'][bb][:n_cal], f['cal']).loc[cal_p].values for bb in BACKBONES}
    lr_te = {bb: pmean(res['lr'][bb][n_cal:], te_slices).loc[te_p].values for bb in BACKBONES}
    cal_f1 = {bb: metrics3(y_cal, lr_cal[bb].argmax(1))['macro_F1'] for bb in BACKBONES}
    bb_sel = max(BACKBONES, key=lambda b: cal_f1[b])

    # stage 2: candidates scored on CN and MCI calibration patients, each with its own tuned T2
    s2_cal = {'hybrid|deep': cal_df['CN_vs_MCI'].loc[cal_p].values}
    s2_te = {'hybrid|deep': te_df['CN_vs_MCI'].loc[te_p].values}
    for key, v in res['s2'].items():
        s2_cal[key] = pmean(v[:n_cal], f['cal']).loc[cal_p].values[:, 0]
        s2_te[key] = pmean(v[n_cal:], te_slices).loc[te_p].values[:, 0]
    nonad = y_cal != AD
    y2 = (y_cal[nonad] == MCI).astype(int)
    s2_score = {}
    for key in s2_cal:
        t2, sc = best_threshold(lambda t: bin_macro_f1(y2, s2_cal[key][nonad], t))
        s2_score[key] = (sc, t2)
    s2_sel = max(s2_cal, key=lambda key: s2_score[key][0])
    T2 = s2_score[s2_sel][1]

    # stage 1: T1 tuned for the macro-F1 of the whole cascade, with stage 2 fixed
    p_ad_cal = cal_df['AD_vs_REST'].loc[cal_p].values
    T1, _ = best_threshold(lambda t: metrics3(y_cal, cascade(p_ad_cal, s2_cal[s2_sel], t, T2))['macro_F1'])

    p_ad_te = te_df['AD_vs_REST'].loc[te_p].values
    two_te = cascade(p_ad_te, s2_te[s2_sel], T1, T2)
    two_def_te = cascade(p_ad_te, s2_te['hybrid|deep'], 0.5, 0.5)

    selection.append({'fold': k + 1, 'stage2_model': s2_sel, 'T1': T1, 'T2': T2, 'single_backbone': bb_sel,
                      'cal_stage2_F1': float(s2_score[s2_sel][0]), 'cal_single_F1': float(cal_f1[bb_sel])})
    for i, pid in enumerate(te_p):
        row = {'fold': k + 1, 'patient': pid, 'true': int(y_te[i]), 'ova': int(ova_te[i]),
               'two_stage': int(two_te[i]), 'two_stage_default': int(two_def_te[i]),
               'single_selected': int(lr_te[bb_sel][i].argmax()),
               'p_AD': float(p_ad_te[i]), 'p_MCI_stage2': float(s2_te[s2_sel][i])}
        for bb in BACKBONES:
            row['lr_' + bb] = int(lr_te[bb][i].argmax())
        rows.append(row)

if MODE == 'slice':
    # patient-level aggregation under the slice split
    agg = pmean(np.concatenate(slice_split_probs), np.concatenate(slice_split_idx))
    for pid, row in agg.iterrows():
        rows.append({'patient': pid, 'true': patient_label[pid], 'ova': int(np.argmax(row.values))})

pred_df = pd.DataFrame(rows)
pred_df.to_csv(os.path.join(OUT_DIR, 'patient_predictions.csv'), index=False)
slice_true = np.concatenate(slice_true)
summary = {'run': RUN, 'mode': MODE, 'n_patients': int(pred_df['patient'].nunique()), 'n_test_slices': int(len(slice_true)),
           'slice': {}, 'patient': {}, 'paired': {}, 'selection': selection}

for name, preds in slice_pred.items():
    pr = np.concatenate(preds)
    p, r, f1, _ = precision_recall_fscore_support(slice_true, pr, labels=[0, 1, 2], zero_division=0)
    summary['slice'][name] = {'acc': float(accuracy_score(slice_true, pr)), 'macro_F1': float(f1.mean()), 'macro_R': float(r.mean())}

models = ['ova'] if MODE == 'slice' else (['ova', 'two_stage', 'two_stage_default', 'single_selected'] +
                                          ['lr_' + bb for bb in BACKBONES])
for name in models:
    m = metrics3(pred_df['true'].values, pred_df[name].values)
    m['macro_F1_CI'], m['macro_R_CI'] = boot_ci(pred_df['true'].values, pred_df[name].values)
    summary['patient'][name] = m

if MODE == 'patient':
    y = pred_df['true'].values
    summary['paired']['two_stage_vs_ova'] = paired(y, pred_df['two_stage'].values, pred_df['ova'].values)
    summary['paired']['two_stage_vs_single'] = paired(y, pred_df['two_stage'].values, pred_df['single_selected'].values)
    summary['paired']['ova_vs_single'] = paired(y, pred_df['ova'].values, pred_df['single_selected'].values)
    summary['stage1_AUC'] = float(roc_auc_score((y == AD).astype(int), pred_df['p_AD'].values))
    nonad = y != AD
    y2 = (y[nonad] == MCI).astype(int)
    p2 = pred_df['p_MCI_stage2'].values[nonad]
    t2 = pred_df['two_stage'].values[nonad]
    summary['stage2_AUC'] = float(roc_auc_score(y2, p2))
    summary['stage2_BA_in_cascade'] = float(np.mean([np.mean(t2[y2 == 0] == CN), np.mean(t2[y2 == 1] == MCI)]))

with open(os.path.join(OUT_DIR, 'metrics.json'), 'w') as fh:
    json.dump(summary, fh, indent=1)

show = pd.DataFrame(summary['patient']).T[['macro_P', 'macro_R', 'macro_F1', 'R_AD', 'R_CN', 'R_MCI']].astype(float).round(3)
print(show.to_string())
print('\nSlice level:', {k: round(v['macro_F1'], 3) for k, v in summary['slice'].items()})
if MODE == 'patient':
    print('Two-stage minus OvA:', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in summary['paired']['two_stage_vs_ova'].items()})

                         macro_P  macro_R  macro_F1   R_AD   R_CN  R_MCI
ova                        0.392    0.374     0.376  0.204  0.239  0.680
two_stage                  0.407    0.391     0.393  0.185  0.307  0.680
two_stage_default          0.427    0.389     0.393  0.185  0.261  0.719
single_selected            0.396    0.397     0.394  0.333  0.295  0.562
lr_resnet18+densenet121    0.418    0.418     0.416  0.333  0.341  0.579
lr_resnet50                0.403    0.416     0.407  0.407  0.295  0.545
lr_densenet121             0.423    0.423     0.421  0.315  0.375  0.579
lr_efficientnet_b0         0.400    0.401     0.400  0.259  0.386  0.556

Slice level: {'ova': 0.378, 'lr_resnet18+densenet121': 0.404, 'lr_resnet50': 0.375, 'lr_densenet121': 0.399, 'lr_efficientnet_b0': 0.381}
Two-stage minus OvA: {'diff': 0.018, 'ci': [-0.03091787937708087, 0.06462927959973039], 'only_a': 32, 'only_b': 27, 'mcnemar_p': 0.603}
